# Notebook 1 — Preparação dos Datasets

Parte do `dataset_original.xlsx` e produz **7 datasets** (um por área térmica) no formato *wide*, aplicando:

1. **Padronização de categorias duplicadas** (grafia, maiúsculas, sinônimos).
2. **Unificação de `housing = "open - updated"` com `"open"`.**
3. **Divisão em 7 datasets** por `Key Area`.

Descarta `Reason`, `TC Number` e `Key Area`. O `Active Limit` sai do treino mas é salvo à parte como tabela de referência.

Outliers, nulos, agregações e one-hot são etapas posteriores (notebooks de tratamento).


## 1. Configuração

Importamos as bibliotecas e definimos onde está o arquivo.

- **`pandas`** (`pd`): a ferramenta principal para trabalhar com tabelas (chamadas *DataFrames*).
- **`numpy`** (`np`): usamos para o `np.nan`, que representa "valor ausente" (célula vazia).
- **`re`** e **`unicodedata`**: para limpar texto (espaços, acentos, maiúsculas).
- **`os`**: para procurar o arquivo em caminhos alternativos, caso não esteja na pasta atual.

In [ ]:
# == caminhos do projeto ==
# A pasta deste caderno, dentro de notebooks/. Os CSVs sao lidos e gravados
# nela; as leituras entre pastas usam ../<outra pasta>. Para que esses caminhos
# valham qualquer que seja o diretorio em que o kernel foi aberto, o bloco
# abaixo localiza a raiz do repositorio -- a pasta que contem notebooks/ e
# documents/ -- e entra na pasta do caderno. No Colab, onde nao existe
# repositorio e os arquivos ficam soltos ao lado do caderno, nada e alterado.
import os

PASTA_CADERNO = "colabs_S4"


def raiz_repositorio(inicio=None):
    """Sobe a arvore de diretorios ate a raiz do repositorio; None se nao achar."""
    atual = os.path.abspath(inicio or os.getcwd())
    while True:
        if all(os.path.isdir(os.path.join(atual, p)) for p in ("notebooks", "documents")):
            return atual
        pai = os.path.dirname(atual)
        if pai == atual:
            return None
        atual = pai


RAIZ_REPO = raiz_repositorio()
if RAIZ_REPO:
    os.chdir(os.path.join(RAIZ_REPO, "notebooks", PASTA_CADERNO))
print("Diretorio de trabalho:", os.path.abspath("."))

In [ ]:
import json
import pandas as pd
import numpy as np
import re
import unicodedata
import os

ABA = 0

# O arquivo pode estar ao lado do notebook, como no Colab, ou na pasta do
# repositorio. O primeiro caminho existente e adotado, de modo que o notebook
# roda em qualquer maquina sem edicao manual.
_CANDIDATOS = [
    "dataset_original.xlsx",
    os.path.join("data", "dataset_original.xlsx"),
    "/content/dataset_original.xlsx",
    os.path.join("..", "..", "documents", "outros", "dataset_original.xlsx"),
    os.path.join("..", "documents", "outros", "dataset_original.xlsx"),
    os.path.join("documents", "outros", "dataset_original.xlsx"),
]
CAMINHO_ORIGINAL = next((c for c in _CANDIDATOS if os.path.exists(c)), None)
if CAMINHO_ORIGINAL is None:
    raise FileNotFoundError(
        "dataset_original.xlsx nao encontrado. Diretorio atual: " + os.path.abspath(".")
        + ". Caminhos tentados: " + "; ".join(_CANDIDATOS))
print("Lendo:", CAMINHO_ORIGINAL)


## 2. Leitura da planilha bruta

A planilha tem um formato **transposto**: os ensaios estão nas colunas e os componentes nas linhas — o inverso do que um modelo espera. Por isso lemos com `header=None` (sem tratar nenhuma linha como cabeçalho) e reconstruímos a estrutura manualmente nas próximas seções.

O resultado, `raw`, é a planilha crua: 97 linhas por 60 colunas, exatamente como está no Excel.

In [ ]:
raw = pd.read_excel(CAMINHO_ORIGINAL, sheet_name=ABA, header=None)
print("Formato bruto:", raw.shape)
raw.iloc[:12, :6]


## 3. Configurações de cada ensaio (linhas 2 a 8)

As linhas 2 a 8 da planilha descrevem a **configuração de design** de cada ensaio (blower, housing, run case, etc.). A linha 1 (`Reason`) é descartada já aqui — ela é texto livre sobre o motivo do teste, não uma variável física que influencie a temperatura.

Cada ensaio recebe um identificador interno `run_idx` (0, 1, 2…). Usamos esse número em vez do nome original ("Run 1", "Run 2"…) porque os nomes se repetem na planilha e não serviriam como chave única. O `for` percorre cada linha de feature e a transforma em uma coluna da tabela `ensaios`.

In [ ]:
LINHAS_FEATURES = {
    2: "blower_direction",
    3: "housing",
    4: "board_position",
    5: "run_case",
    6: "losses",
    7: "deflector_hole",
    8: "material_coil_support",
}
n_ensaios = raw.shape[1] - 4

ensaios = pd.DataFrame({"run_idx": range(n_ensaios)})
for linha, nome in LINHAS_FEATURES.items():
    valores = raw.iloc[linha, 4:].tolist()
    ensaios[nome] = [str(v).strip() if pd.notna(v) else np.nan for v in valores]

print("Ensaios:", len(ensaios))
ensaios.head()


## 4. Padronização das categorias duplicadas

O mesmo valor às vezes aparece escrito de formas diferentes, e o computador trataria cada grafia como uma categoria distinta. A padronização acontece em dois passos:

1. **Normalização geral** (`normalizar`): converte para minúsculas, remove acentos e espaços duplicados. Isso resolve a maioria dos casos, como `To Front` / `To front` → `to front` e `To Back` / `To back` → `to back`.
2. **Mapa de sinônimos** (`SINONIMOS`): unifica nomes diferentes para a mesma coisa, como `Broil Case 2` → `broil c2`.

O `.map()` aplica a função a cada valor da coluna; o `.replace()` troca os sinônimos. Os nomes são mantidos como no original (só padronizados), **sem tradução**, e os nulos são preservados como `NaN`.

In [ ]:
def normalizar(texto):
    if pd.isna(texto):
        return np.nan
    texto = unicodedata.normalize("NFKD", str(texto)).encode("ascii", "ignore").decode()
    return re.sub(r"\s+", " ", texto).strip().lower()

COLUNAS_CATEGORICAS = list(LINHAS_FEATURES.values())
for col in COLUNAS_CATEGORICAS:
    ensaios[col] = ensaios[col].map(normalizar)

SINONIMOS = {
    "run_case": {"broil case 2": "broil c2", "broil qual": "broil qualitative",
                 "qualitative": "broil qualitative"},
    "material_coil_support": {"aluminium plate": "aluminium"},
}
for coluna, mapa in SINONIMOS.items():
    ensaios[coluna] = ensaios[coluna].replace(mapa)

for col in COLUNAS_CATEGORICAS:
    print(col, "->", sorted(ensaios[col].dropna().unique().tolist()))


## Atributos físicos extraídos do motivo do ensaio

A linha do motivo é descartada como variável categórica, e com razão: são 35 valores distintos em 56 ensaios, de modo que cada rótulo quase identifica o próprio ensaio. O texto, contudo, descreve alterações físicas de projeto que não constam de nenhuma outra linha da planilha, entre elas a intensidade do soprador, a presença de cobertura sobre o dissipador, a alteração do duto e a troca da bandeja.

A extração é justificada por evidência anterior à sua realização. A investigação dos valores extremos, registrada no notebook de modelagem, verificou que 78 das 92 leituras extremas da base ocorrem em ensaios do modo pyro cujo motivo declara variação de ventilação: o modelo vinha sendo cobrado por prever um efeito cuja causa não lhe era fornecida.

### Critérios adotados

Os rótulos que nomeiam o próprio ensaio, como Run 31, não geram atributo algum: eles reintroduziriam a identidade do ensaio como variável, precisamente o que a Seção 4.2.2.3 da documentação proíbe.

O suporte é conferido por nível, e não por coluna. A distinção importa: a intensidade do soprador reúne nove ensaios que a declaram, o que passaria em uma conferência feita sobre a coluna inteira, mas o nível baixa aparecia em um único ensaio. Um nível presente em um só ensaio funciona como identificador desse ensaio, e força o modelo a extrapolar sobre um valor que o treino nunca observa.

A codificação adotada é de indicadores binários, e não de escala ordinal. Uma escala ordinal precisaria atribuir algum valor ao caso não declarado, e qualquer valor escolhido afirma algo falso: o zero coloca o desconhecido abaixo do nível mais baixo, declarando que um ensaio sem menção ao soprador possui menos vazão que um ensaio com soprador em baixa. Dois indicadores substituem a escala sem afirmar nada por omissão, separando o declarado do silêncio e ordenando o que foi declarado.

In [ ]:
# Linha 1 da planilha: motivo declarado de cada ensaio.
motivos = [str(v).strip().lower() if pd.notna(v) else "" for v in raw.iloc[1, 4:].tolist()]

SUPORTE_MINIMO = 3   # ensaios necessarios para cada nivel de um atributo


def declara_vazao(t):
    """Se o motivo menciona a intensidade do soprador, em qualquer nivel."""
    return int("flow" in t or "blower on" in t or "fan on" in t
               or "cfm" in t or "rpm" in t)


def vazao_alta(t):
    """Se a intensidade declarada e a mais alta da escala."""
    return int("double flow" in t or "more flow" in t
               or "blower on high" in t or "16 cfm" in t)


def suporte_por_nivel(serie):
    """Menor numero de ensaios entre os niveis que nao sao o de referencia."""
    contagem = serie[serie != 0].value_counts()
    return int(contagem.min()) if len(contagem) else 0


derivados = pd.DataFrame({
    "cfg_vazao_declarada": [declara_vazao(t) for t in motivos],
    "cfg_vazao_alta": [vazao_alta(t) for t in motivos],
    # Ausencia de cobertura e ausencia de mencao sao reunidas no zero: nos dois
    # casos nao ha cobertura declarada.
    "cfg_cobertura_dissipador": [int("added a cover" in t or "with cover" in t)
                                 for t in motivos],
    "cfg_duto_alterado": [int("duct" in t) for t in motivos],
    "cfg_bandeja_alterada": [int("tray" in t) for t in motivos],
    # "run " no texto indica rotulo que nomeia o ensaio, e nao alteracao fisica
    "cfg_deflector_variante": [int("deflector" in t and "run " not in t) for t in motivos],
    "cfg_sem_cooktop": [int("wo cookitop" in t) for t in motivos],
    "cfg_espacador": [int("spacer" in t) for t in motivos],
})

print("Suporte de cada atributo, pelo nivel de menor suporte:")
mantidos = []
for col in derivados.columns:
    suporte = suporte_por_nivel(derivados[col])
    ok = suporte >= SUPORTE_MINIMO
    print(f"  {col:26s} {suporte:2d} ensaios  {'mantido' if ok else 'DESCARTADO'}")
    if ok:
        mantidos.append(col)

if "cfg_vazao_alta" in mantidos and "cfg_vazao_declarada" in mantidos:
    incoerentes = derivados[(derivados["cfg_vazao_alta"] == 1)
                            & (derivados["cfg_vazao_declarada"] == 0)]
    assert len(incoerentes) == 0, "vazao alta sem vazao declarada"

# A tabela completa dos oito indicadores permanece, porque e ela que sustenta a
# chave de replicata gravada adiante. A chave precisa da distincao mais fina
# disponivel: dois ensaios so devem receber a mesma chave quando nada os separa.
# O modelo, ao contrario, recebe o motivo agrupado, montado logo abaixo.
derivados_completo = derivados.copy()
mantidos = []


def familia_do_motivo(t):
    """Uma familia por ensaio, mutuamente exclusivas, pela alteracao dominante.

    Os seis indicadores anteriores nao formavam particao: um ensaio com cobertura
    e duto novo acendia duas colunas, e o modelo precisava aprender a combinacao
    em lugar de ler um rotulo. Medido sob o protocolo do projeto, o agrupamento
    reduz o erro de 6,670 para 6,146 graus com floresta e gradiente, ganho de
    0,524 com intervalo de 0,384 a 0,670 e nas tres sementes, e ao mesmo tempo
    reduz as aprovacoes indevidas.

    A ordem das condicoes importa: prevalece o caminho do ar sobre o material, e
    o material sobre a rotulagem de linha de base.
    """
    if not t or t == "nan":
        return "not_declared"
    if "study of position" in t:
        return "position_study"
    if ("cfm" in t or "flow" in t or "rpm" in t or "fan " in t or "blower" in t
            or "duct" in t or "deflector" in t or "tray" in t):
        return "air_path"
    if "cover" in t and ("heatsink" in t or "hs" in t or t.startswith("run with cover")):
        return "heatsink_cover"
    if "plate" in t or "spacer" in t:
        return "plate"
    return "other"


_familia = pd.Series([familia_do_motivo(t) for t in motivos])
NIVEIS_MOTIVO = ["air_path", "heatsink_cover", "plate", "position_study",
                 "other", "not_declared"]
assert set(_familia) <= set(NIVEIS_MOTIVO), "familia fora dos niveis declarados"

# Os niveis formam particao, de modo que o filtro de suporte minimo nao se
# aplica: descartar um nivel apenas o tornaria a categoria de referencia, e nada
# se perderia nem se ganharia com isso.
print("Motivo declarado, ensaios por familia:")
for nivel in NIVEIS_MOTIVO:
    col = f"reason_{nivel}"
    ensaios[col] = (_familia == nivel).astype(int).to_numpy()
    mantidos.append(col)
    print(f"  {col:26s} {int((_familia == nivel).sum()):2d} ensaios")

print()
print("Atributos incorporados:", len(mantidos))

### Ordem de acionamento, extraída do rótulo de perda

O rótulo da coluna de perdas é composto: além da família do perfil, ele registra a ordem de chaveamento dos IGBTs, com os termos front order e back order, conforme o ensaio de posição 30 do dado bruto, descrito como Diff Order IGBTs. Na codificação por perfil essa informação era descartada, e na codificação por treze categorias ficava dissolvida em níveis com um único ensaio.

A ordem é extraída aqui como atributo próprio, com quatro níveis: front, back, não declarada, e não se aplica, este último para os ensaios sem perda e sem rótulo. O prefixo perfil_perdas já consta das listas de entradas dos notebooks seguintes, de modo que as colunas são reconhecidas como entradas sem edição adicional. A inclusão foi medida sob validação por configuração e reduz o erro em 0,118 grau no agregado, com intervalo de confiança inteiramente acima de zero.

In [ ]:
def ordem_de_acionamento(rotulo):
    """Nivel da ordem de chaveamento declarado no rotulo de perda."""
    t = str(rotulo).strip().lower() if pd.notna(rotulo) else ""
    if t == "" or "no losses" in t:
        return "not_applicable"
    if "front order" in t:
        return "front"
    if "back order" in t:
        return "back"
    return "not_declared"


# Os nomes seguem o restante das colunas, em ingles. Registre-se que estas quatro
# colunas foram medidas como prejudiciais: a base sobe de 6,646 para 6,705 graus
# quando elas entram, e a auditoria por atribuicao lhes da 1,2 por cento em
# media, nula em cinco das sete regioes. Elas permanecem para que a decisao de
# remove-las seja tomada com a medicao a vista.
_ordem = ensaios["losses"].map(ordem_de_acionamento)
ORDEM_NIVEIS = ["front", "back", "not_declared", "not_applicable"]
for nivel in ORDEM_NIVEIS:
    col = f"loss_profile_order_{nivel}"
    ensaios[col] = (_ordem == nivel).astype(int).to_numpy()
    mantidos.append(col)

print("Ordem de acionamento, suporte por nivel:")
print(_ordem.value_counts().to_string())
print()
print("Atributos incorporados, com a ordem:", len(mantidos))

## 5. Unificação de `housing`: "open - updated" → "open"

Análise prévia mostrou que a diferença de temperatura entre `open` e `open - updated` **inverte de sinal** conforme o `run_case` (em broil c2 o "open - updated" é mais quente; em broil qualitative é mais frio). Se fosse uma configuração física distinta, a direção seria consistente. A inversão indica que a diferença vem da distribuição desigual dos ensaios, não da geometria. Por isso, unificamos as duas.

Esta é uma decisão de **padronização de categoria** (dois rótulos para a mesma configuração), por isso fica aqui no notebook 1.


In [ ]:
n = (ensaios["housing"] == "open - updated").sum()
ensaios["housing"] = ensaios["housing"].replace({"open - updated": "open"})
print(f"{n} ensaios com 'open - updated' unificados em 'open'")
print("Categorias finais de housing:", sorted(ensaios["housing"].dropna().unique().tolist()))


## 6. Componentes e temperaturas (linhas 11 em diante)

As linhas 11 em diante trazem os componentes medidos. As 4 primeiras colunas são `TC Number`, `TC Name`, `Key Area` e `Active Limit`; as demais são as temperaturas por ensaio.

Três cuidados:
- **Remover linhas de seção** (títulos como "Mainback Components") que não têm `TC Name`.
- **`ffill()` no `Key Area`**: "forward fill" copia o valor da célula de cima quando está vazia. Isso resolve as "Aluminium Plate", que têm área em branco na planilha mas pertencem a Coil Temps.
- **Limpar espaços** nos nomes dos componentes (sem traduzi-los).

In [ ]:
componentes = raw.iloc[10:93].copy()
componentes.columns = ["tc_number", "tc_name", "key_area", "active_limit"] + list(range(n_ensaios))
componentes = componentes[componentes["tc_name"].notna()].copy()
componentes["key_area"] = componentes["key_area"].ffill()
componentes["tc_name"] = (componentes["tc_name"].astype(str)
                          .str.replace(r"\s+", " ", regex=True).str.strip())
print("Componentes:", len(componentes))
print("Áreas:", sorted(componentes["key_area"].unique().tolist()))


## 7. Tabela de referência do `Active Limit`

O `Active Limit` não entra no treino — ele serve para, **depois** da predição, comparar a temperatura prevista com o limite de segurança do componente (zona segura / no limite / excedida).

Guardamos numa tabela à parte, um limite por componente. A expressão regular extrai o número de dentro do texto (alguns limites vêm como "Monitor (150 C)"), e `errors="coerce"` transforma em `NaN` os casos sem número (componentes que são apenas de monitoramento, marcados em `apenas_monitorar`).

In [ ]:
limites = componentes[["tc_name", "key_area", "active_limit"]].copy()
limites["limite_num"] = pd.to_numeric(
    limites["active_limit"].astype(str).str.extract(r"(\d+\.?\d*)")[0], errors="coerce")
limites["apenas_monitorar"] = limites["limite_num"].isna()
limites.to_csv("limites_ativos.csv", index=False)
print("Salvo: limites_ativos.csv")
limites.head()


## 8. Formato longo e junção com as configurações

`.melt()` transforma a tabela **larga** (uma coluna por ensaio) em **longa** (uma linha por combinação componente×ensaio). É o passo que "desempilha" a matriz de temperaturas.

Em seguida, `pd.to_numeric` converte as temperaturas em número (valores não numéricos viram `NaN`), e `.merge()` junta as configurações de cada ensaio a cada leitura, usando `run_idx` como chave. No fim, cada linha tem: o componente, sua área, a temperatura e as 7 features de design daquele ensaio.

In [ ]:
longo = componentes.melt(id_vars=["tc_name", "key_area"], value_vars=list(range(n_ensaios)),
                         var_name="run_idx", value_name="temperatura_c")
longo["temperatura_c"] = pd.to_numeric(longo["temperatura_c"], errors="coerce")
longo = longo.merge(ensaios, on="run_idx", how="left")
print("Formato longo:", longo.shape)
longo.head()


## 9. Divisão nos 7 datasets (wide)

`.groupby("key_area")` separa a tabela em 7 grupos (um por área). Para cada área, `.pivot_table()` remonta o formato **wide**: cada ensaio vira uma linha, cada componente vira uma coluna de temperatura. A `Key Area` não entra nos arquivos — ela é o critério que os separa.

`dropna=False` faz com que componentes sem nenhuma medição apareçam como coluna inteiramente nula, em vez de sumirem em silêncio (a remoção deles é feita, de forma registrada, no notebook de tratamento). Cada área é salva em seu próprio CSV.

In [ ]:
ORDEM_FEATURES_BASE = ["blower_direction", "housing", "board_position", "run_case",
                       "losses", "deflector_hole", "material_coil_support"]
ORDEM_FEATURES = ORDEM_FEATURES_BASE + mantidos

def slug(texto):
    return re.sub(r"[^a-z0-9]+", "_", texto.lower()).strip("_")

resumo = []
for area, grupo in longo.groupby("key_area"):
    alvos = grupo.pivot_table(index="run_idx", columns="tc_name",
                              values="temperatura_c", aggfunc="first", dropna=False)
    # Componentes identicos bit a bit sao descartados. Na base atual, Microchip e
    # Thermistor da interface tem correlacao 1,0 e diferenca maxima zero nas 45
    # leituras: ou e o mesmo sensor registrado duas vezes, ou houve copia. Mantidos
    # os dois, a leitura entra em dobro no treino, na validacao e na media das
    # metricas, como se fosse medicao independente. Qual dos dois nomes corresponde
    # ao sensor real e pergunta para a parceira; aqui preserva-se o primeiro.
    vistos, duplicados = {}, []
    for col in alvos.columns:
        chave = tuple(alvos[col].fillna(-9e9).tolist())
        if chave in vistos:
            duplicados.append((col, vistos[chave]))
        else:
            vistos[chave] = col
    if duplicados:
        alvos = alvos.drop(columns=[c for c, _ in duplicados])
        for c, orig in duplicados:
            print(f"  {area}: descartado {c!r}, identico a {orig!r}")

    design = grupo.drop_duplicates("run_idx").set_index("run_idx")[ORDEM_FEATURES]
    wide = design.join(alvos).reset_index()
    nome = f"dataset_{slug(area)}.csv"
    wide.to_csv(nome, index=False)
    resumo.append({"key_area": area, "arquivo": nome,
                   "ensaios": wide.shape[0], "componentes": alvos.shape[1]})

pd.DataFrame(resumo).sort_values("componentes", ascending=False)


---
### Resultado
7 arquivos `dataset_<area>.csv` + `limites_ativos.csv`, padronizados, com housing unificado e divididos. Prontos para os notebooks de tratamento.


## Chave de replicata

O piso de erro da base é estimado pela discordância entre ensaios de configuração idêntica, e depende, portanto, de uma definição de configuração idêntica. Essa definição não pode ser o vetor de entrada do modelo: a codificação reúne deliberadamente níveis de suporte insuficiente, de modo que dois ensaios fisicamente distintos podem apresentar o mesmo vetor, e a diferença entre eles seria contabilizada como ruído.

É gravada aqui, por esse motivo, uma chave construída sobre a descrição mais fina disponível de cada ensaio: as sete variáveis de configuração, os oito atributos candidatos anteriores ao filtro de suporte, e o próprio texto do motivo normalizado. Ela não constitui variável de entrada em hipótese alguma, e é lida apenas pelo cálculo do piso, no notebook de modelagem.

In [ ]:
_motivo_norm = [re.sub(r"[^a-z0-9]+", " ", t).strip() for t in motivos]

# Os cabecalhos das colunas do dado bruto numeram os pontos de um planejamento de
# experimentos sobre a posicao do soprador. A posicao nao consta de nenhuma
# coluna, de modo que dois pontos desse planejamento recebiam a mesma chave e
# eram contados como repeticao, inflando o piso de ruido. Quando o motivo
# declara o estudo de posicao, o indice do planejamento passa a integrar a chave.
# O bruto e lido sem cabecalho: os titulos das colunas estao na linha 0.
_cabecalhos = [str(c).lower() for c in raw.iloc[0, 4:].tolist()]


# Por que o marcador existe: os ensaios 16 e 17 sao dois pontos de um estudo de
# posicao do soprador, e a posicao nao consta de nenhuma coluna. Sem o marcador
# os dois recebiam a mesma chave e passavam por repeticao, o que inflava o piso
# de ruido. Separando o par, o piso cai de 6,36 para 5,07 graus nas bobinas, de
# 1,33 para 0,57 na tampa traseira e de 2,15 para 0,87 na refrigeracao da placa.
def _marcador_doe(pos):
    if "study of position" not in _motivo_norm[pos]:
        return ""
    m = re.search(r"doe run (\d+)", _cabecalhos[pos])
    return f"doe{m.group(1)}" if m else ""


chave_replicata = {}
for pos, run in enumerate(ensaios["run_idx"]):
    partes = [str(ensaios.iloc[pos][c]) for c in ORDEM_FEATURES_BASE]
    partes += [str(int(derivados_completo.iloc[pos][c]))
               for c in derivados_completo.columns]
    partes.append(_motivo_norm[pos])
    partes.append(_marcador_doe(pos))
    chave_replicata[int(run)] = "|".join(partes)

with open("chave_replicata.json", "w", encoding="utf-8") as f:
    json.dump({"chave_por_run_idx": chave_replicata,
               "n_ensaios": len(chave_replicata),
               "n_configuracoes": len(set(chave_replicata.values()))},
              f, ensure_ascii=False, indent=1)

_cont = pd.Series(list(chave_replicata.values())).value_counts()
print("Salvo: chave_replicata.json")
print(f"Configuracoes distintas: {len(_cont)} de {len(chave_replicata)} ensaios")
print(f"Configuracoes com repeticao: {int((_cont > 1).sum())}, "
      f"cobrindo {int(_cont[_cont > 1].sum())} ensaios")